In [142]:
from __future__ import annotations

import pandas as pd
import numpy as np
from bokeh.plotting import figure, curdoc, show, output_file, output_notebook
from bokeh.models import Button, ColumnDataSource, Slider, Select, Span, CustomJS, CheckboxGroup, ResetTool, RadioGroup, \
    ButtonGroup, TapTool, BoxZoomTool, Span, HoverTool, SaveTool
import bokeh.layouts
import bokeh.palettes
from bokeh.layouts import gridplot
from pathlib import Path, WindowsPath
from skimage.io import imread
import ast
import scipy

import typing

# Turn off warnings
import logging  # isort:skip

log = logging.getLogger(__name__)

from bokeh.util.warnings import BokehUserWarning
import warnings

warnings.simplefilter(action='ignore', category=BokehUserWarning)


def normalize_df(dataframe):
    """ 0-1 normalization of the entire dataframe

    Parameters:
        dataframe: pandas.DataFrame
            Input dataframe

    Returns:
        normalized_dataframe: pandas.DataFrame
            Pandas dataframe containing the 0-1 normalization for each column

    """
    normalized_dataframe = dataframe.copy(deep=True)
    for column in normalized_dataframe.columns.values:
        normalized_dataframe[column] = (normalized_dataframe[column] - normalized_dataframe[column].min()) / (
                    normalized_dataframe[column].max() - normalized_dataframe[column].min())
    return normalized_dataframe


def get_abs(signal, peak_heights, min_max_values):
    """ Revert 0-1 normalization

    Parameters:
        signal: str
            The name of the cell signal, as used in the traces dataframe
        peak_heights: list
            List containing the normalized peak intensities
        min_max_values: dictionary
            A dictionary with key-values pairs being the keys the name of the cells and
            the values a list with the first element being the minimum intensity for that
            particular trace and the second element the maximum intensity for that trace

    Returns:
        abs_heights: list
            A list of the absolute intensities of each peak

    """

    min_value = min_max_values[signal][0]
    max_value = min_max_values[signal][1]
    abs_heights = []
    for height in peak_heights:
        abs_height = height * (max_value - min_value) + min_value
        abs_heights.append(abs_height)

    return abs_heights


def dashboard_registration(traces_path: typing.Type[Path],
                           exp_name: str,
                           contours_path: str,
                           notebook_url: str = 'localhost:8888',
                           interval: typing.Union[int, float] = 11,
                           bf_folder_path: str = './bf_images/') -> None:
    """

    """
    # App to record peaks
    # Implement manual addition of peaks - removal still to be added
    # Implement sliders for beginning of oscillations, end of oscillation
    # Implement a button to change the validity of the signal
    # Turn off warnings

    #     from __future__ import annotations

    #     import logging # isort:skip
    #     log = logging.getLogger(__name__)

    #     from bokeh.util.warnings import BokehUserWarning
    #     import warnings
    #     warnings.simplefilter(action='ignore', category=BokehUserWarning)

    # Validity of a given signal
    global validity
    validity = True

    # Show contour
    global contour
    contour = True

    # Display BF
    global display
    display = True

    # SInitial values
    signals = pd.read_csv(traces_path.as_posix(), index_col=0)
    all_signals_norm = normalize_df(signals)
    min_max_values = {str(cell): [signals[cell].min(), signals[cell].max()] for cell in signals.columns}
    signals_df = all_signals_norm.copy(deep=True)
    t = np.linspace(0, len(signals_df) - 1, len(signals_df))
    t = [int(i) for i in t]
    selected_signal = signals_df.columns.values[0]
    signal = signals_df[selected_signal]

    peaks = []
    heights = []
    troughs = []
    trough_heights = []

    # Initial parameters for find_peaks
    initial_prominence = 0.5
    initial_height = 0.01
    threshold = np.zeros(len(t))

    # Initial parameters for timepoint sliders
    initial_tf = 0
    v_line = np.linspace(0, len(signals_df) - 1, len(signals_df))
    tf = np.ones(len(v_line))
    bo = np.zeros(len(v_line))
    eo = np.zeros(len(v_line))
    
    # Load and display initial image
    image_path = Path(bf_folder_path + selected_signal[:-2] + '.tif')
    image_stack = imread(image_path.as_posix())[:, :, :]
    img_data = {'img': [image_stack[initial_tf]], 'img_stack': [image_stack], 'path': [image_path.as_posix()]}

    # Load contour data and plot first one
    contour_data = pd.read_csv(contours_path, index_col='Time Frame').iloc[:,1:]
    contour_dict = {
        'x_coords': ast.literal_eval(
            contour_data.loc[(contour_data['Cell Name'] == selected_signal), 'x_coords'][initial_tf]),
        'y_coords': ast.literal_eval(
            contour_data.loc[(contour_data['Cell Name'] == selected_signal), 'y_coords'][initial_tf]), }
       
    # Data dataframe to store data
    global peak_data_dashboard
    peak_data_dashboard = pd.DataFrame()

    # Find Peaks functions
    # def find_peaks_with_params(signal, prominence, height):
    #     peaks, properties = scipy.signal.find_peaks(signal, prominence=prominence, height=height)
    #     return peaks, properties['peak_heights']

    def find_peaks_with_params(signal, prominence, height):
        peaks, properties = scipy.signal.find_peaks(signal, prominence=prominence, height=height)
        properties = signal.iloc[peaks]
        return peaks, properties

    def find_troughs_with_params(signal, prominence):
        troughs, properties = scipy.signal.find_peaks(-signal, prominence=prominence)
        properties = signal.iloc[troughs]
        return troughs, properties

    # Create ColumnDataSource
    source1 = ColumnDataSource(data=dict(t=t, signal=signal))  # Signal and time domain
    source2 = ColumnDataSource(data=dict(peaks=[], heights=[]))  # Peaks and peak heights
    source3 = ColumnDataSource(data=dict(t=t, threshold=threshold))  # Time domain and height threshold
    source4 = ColumnDataSource(data=dict(troughs=troughs, trough_heights=trough_heights))  # Troughs and heights
    source5 = ColumnDataSource(data=dict(tf=tf, v_line=v_line))  # Time frame and vertical line
    source6 = ColumnDataSource(data=contour_dict)
    source7 = ColumnDataSource(data=dict(bo=bo, v_line=v_line))  # Beginning of oscillations
    source8 = ColumnDataSource(data=dict(eo=eo, v_line=v_line))  # End of oscillations
    source10 = ColumnDataSource(data=img_data)  # Image data for BF display

    # Create Bokeh figure to display signal and peaks
    plot = figure(title='Normalized ' + signals_df.columns.values[0],
                  x_range=(np.min(t) - 0.3, np.max(t) + 0.3), y_range=(np.min(signal) - 0.025, np.max(signal) + 0.025),
                  width=1000, height=800, tools=[TapTool(), BoxZoomTool(), ResetTool()])
    plot.circle('t', 'signal', source=source1, line_width=2, size=6, line_color='blue', legend_label='Signal',
                nonselection_alpha=1.0)
    plot.line('t', 'signal', source=source1, line_width=2, line_color='blue', legend_label='Signal',
              nonselection_alpha=1.0)
    plot.circle('peaks', 'heights', source=source2, size=8, color='red', legend_label='Peaks', line_width=2,
                line_color='black', nonselection_alpha=1.0)
    plot.circle('troughs', 'trough_heights', source=source4, size=8, color='green', legend_label='Trophs', line_width=2,
                line_color='black', nonselection_alpha=1.0)
    plot.line('t', 'threshold', source=source3, line_color='green', line_dash='dashed', legend_label='Height threshold')
    tod_renderer = plot.line('tf', 'v_line', source=source5, line_color='red', line_width=4, line_dash='solid',
                             legend_label='Time of death')
    plot.line('bo', 'v_line', source=source7, line_color='orange', line_dash='dashed', line_width=3,
              legend_label='Beginning of oscillations')
    plot.line('eo', 'v_line', source=source8, line_color='blue', line_dash='dashed', line_width=3,
              legend_label='End of oscillations')

    # Create Bokeh figure to display BF images
    if display == True:
        bf_display = figure(x_range=(0, image_stack.shape[1]), y_range=(0, image_stack.shape[2]))
        # bf_display = figure(x_range=(0, image_stack.shape[1]), y_range=(0, image_stack.shape[2]), width=1000, height=800)
        bf_display.image(image='img', x=0, y=0, dw=image_stack.shape[1], dh=image_stack.shape[2], source=source10,
                         palette='Greys256')

        if contour == True:
            try:
                contour_renderer = bf_display.circle('x_coords', 'y_coords', source=source6, line_color='orange',
                                                    line_dash='solid', legend_label='Contour', size=1)
            # Handle invalid contours - e.g. nan for bad segmentation
            except KeyError:
                pass

    # Callback function for dropdown menu
    def update_signal(attr, old, new):
        selected_signal = signal_select.value
        signal = signals_df[selected_signal]

        # Update data source
        source1.data = dict(t=t, signal=signal)

        # Update peaks and heights based on new signal
        prominence_value = prominence_slider.value
        height_value = height_slider.value
        peaks, peak_heights = find_peaks_with_params(signal, prominence_value, height_value)
        troughs, trough_heights = find_troughs_with_params(signal, prominence_value)
        source2.data = dict(peaks=peaks, heights=peak_heights)
        source4.data = dict(troughs=troughs, trough_heights=trough_heights)

        # Update ToD line - To check
        # index = bo_slider.value
        # index = tod[selected_signal].to_numpy().item()
        # tod_list = np.ones(len(v_line)) * index
        # source5.data = dict(tod_list=tod_list, v_line=v_line)
        tf_slider.value = 0
        source5.data['tf'] = np.zeros(len(source5.data['v_line']))

        # Update plot title
        plot.title.text = 'Normalized '+selected_signal

        # Update BF display
        image_path = Path(bf_folder_path + selected_signal[:-2] + '.tif')
        image_stack = imread(image_path.as_posix())[:, :, :]
        img_data = {'img': [image_stack[0]], 'img_stack': [image_stack], 'path': [image_path.as_posix()]}
        source10.data = img_data

        # Update contour
        if display == True and contour == True:
            source6.data['x_coords'] = ast.literal_eval(
                (contour_data.loc[(contour_data['Cell Name'] == signal_select.value), 'x_coords'])[0])
            source6.data['y_coords'] = ast.literal_eval(
                (contour_data.loc[(contour_data['Cell Name'] == signal_select.value), 'y_coords'])[0])

    # Callback function for sliders using on_change
    def update_peaks(attr, old, new):
        # Retrieve values from sliders
        prominence_value = prominence_slider.value
        height_value = height_slider.value
        selected_signal = signal_select.value
        signal = signals_df[selected_signal]
        bo = bo_slider.value
        eo = eo_slider.value
        # Update peaks and heights based on new prominence value
        # First use eo as end limit for find peaks, then use boolean array to remove any peak before bo and finally update threshold
        # Caution here: the behaviour of find peaks is different for beginning and end filtering of peaks
        if eo < len(signal):
            peaks, peak_heights = find_peaks_with_params(signal[:eo + 1], prominence_value, height_value)
            troughs, trough_heights = find_troughs_with_params(signal[:eo + 1], prominence_value)
        else:
            peaks, peak_heights = find_peaks_with_params(signal, prominence_value, height_value)
            troughs, trough_heights = find_troughs_with_params(signal, prominence_value)
        bo_mask = peaks >= bo
        # eo_mask = peaks <= eo
        peaks = peaks[bo_mask]
        peak_heights = peak_heights[bo_mask]
        bo_mask = troughs >= bo
        troughs = troughs[bo_mask]
        trough_heights = trough_heights[bo_mask]
        threshold = np.ones(len(t)) * height_value
        # Update data source
        source2.data = dict(peaks=peaks, heights=peak_heights)
        source3.data = dict(t=t, threshold=threshold)
        source4.data = dict(troughs=troughs, trough_heights=trough_heights)

    # Callback function for vertical slider
    def update_tf(attr, old, new):
        # Neccessary a check to change the signal only if a different signal in the dropdown menu is selected!!!
        # Update line in signal plot
        index = tf_slider.value
        tf = np.ones(len(v_line)) * index
        source5.data = dict(tf=tf, v_line=v_line)
        selected_time_lapse = signal_select.value
        image_path = Path(bf_folder_path + selected_time_lapse[:-2] + '.tif').as_posix()
        if image_path != source10.data['path'][0]:
            image_path = Path(bf_folder_path + selected_time_lapse[:-2] + '.tif').as_posix()
            image_stack = imread(image_path)[:, :, :]
        else:
            image_stack = source10.data['img_stack'][0]

        # Update tf and contour in BF display
        if display == True:
            new_image = image_stack[index]
            source10.data['img'] = [new_image]
            if contour == True:
                try:
                    source6.data['x_coords'] = ast.literal_eval(
                        (contour_data.loc[(contour_data['Cell Name'] == signal_select.value), 'x_coords'])[index])
                    source6.data['y_coords'] = ast.literal_eval(
                        (contour_data.loc[(contour_data['Cell Name'] == signal_select.value), 'y_coords'])[index])
                except IndexError:
                    pass

    # Callback for beginning of oscillations
    def update_bo(attr, old, new):
        index = bo_slider.value
        bo = np.ones(len(v_line)) * index
        source7.data = dict(bo=bo, v_line=v_line)

    # Callback for end of oscillations
    def update_eo(attr, old, new):
        index = eo_slider.value
        eo = np.ones(len(v_line)) * index
        source8.data = dict(eo=eo, v_line=v_line)

    # Save data function
    def save_data():
        selected_signal = signal_select.value
        signal = signals_df[selected_signal]
        prominence_value = prominence_slider.value
        height_value = height_slider.value
        bo = bo_slider.value
        eo = eo_slider.value
        tf = tf_slider.value
        # peaks, peak_heights = find_peaks_with_params(signal, prominence_value, height_value)
        peaks, peak_heights = source2.data['peaks'], source2.data['heights']
        troughs, trough_heights = source4.data['troughs'], source4.data['trough_heights']
        abs_heights = get_abs(signal=selected_signal, peak_heights=source2.data['heights'],
                              min_max_values=min_max_values)
        abs_trough_heights = get_abs(signal=selected_signal, peak_heights=source4.data['trough_heights'],
                                     min_max_values=min_max_values)

        critical_points = np.concatenate([peaks, troughs])
        critical_heights = np.concatenate([peak_heights, trough_heights])
        abs_critical_heights = np.concatenate([abs_heights, abs_trough_heights])
        is_peak = np.concatenate([np.ones(len(peaks)), np.zeros(len(troughs))]).astype(bool)
        is_trough = np.concatenate([np.zeros(len(peaks)), np.ones(len(troughs))]).astype(bool)

        global peak_data_dashboard
        try:
            if peak_data_dashboard['Signal'].isin(
                    [selected_signal]).any():  # Remove the stored-data if the signal is re-done
                peak_data_dashboard = peak_data_dashboard[~peak_data_dashboard['Signal'].isin([selected_signal])]

        except KeyError:
            print('First signal')

        if validity == True:
            data = {
                'Exp': [exp_name] * len(critical_points),
                'Signal': [selected_signal] * len(critical_points),
                'Critical Point pos': critical_points,
                'Critical Point heights': critical_heights,
                'Absolute Critical Point heights': abs_critical_heights,
                'Threshold': [height_value] * len(critical_points),
                'Critical Point proms': [prominence_value] * len(critical_points),
                'Beginning of oscillation': [bo] * len(critical_points),
                'End of oscillation': [eo] * len(critical_points),
                'ToD': [tf] * len(critical_points),
                'Sample rate': [interval] * len(critical_points),
                'Is peak': is_peak,
                'Is trough': is_trough,
            }
        else:
            data = {
                'Exp': [exp_name],
                'Signal': [selected_signal],
                'Critical Point pos': [np.nan],
                'Critical Point heights': [np.nan],
                'Absolute Critical Point heights': [np.nan],
                'Threshold': [np.nan],
                'Critical Point proms': [np.nan],
                'Beginning of oscillation': [np.nan],
                'End of oscillation': [np.nan],
                'ToD': [tf],
                'Sample rate': [interval],
                'Is peak': False,
                'Is trough': False,
            }
        df = pd.DataFrame(data)
        df = df.sort_values(by='Critical Point pos')
        peak_data_dashboard = pd.concat([peak_data_dashboard, df], ignore_index=True)

        # Move to the next signal in the dropdown menu
        signal_index = list(signals_df.keys()).index(selected_signal)
        try:
            # next_index = (signal_index + 1) % len(signals_df)
            next_index = signal_index + 1
            next_signal = list(signals_df.keys())[next_index]
            signal_select.value = next_signal
        except IndexError:
            print('{selected_signal} is the last signal of the dataset'.format(selected_signal=selected_signal))

    # Callback function to toggle BF display
    def callback_display(attr, new, old):
        global display
        display = not display
        bf_display.visible = display
        tf_slider.visible = display
        checkbox_contour.visible = display
        tod_renderer.visible = display

    def callback_contour(attr, new, old):
        global contour
        contour = not contour
        contour_renderer.visible = contour

    def tap_point(attr, old, new):
        try:
            peaks = list(source2.data['peaks'])
            peak_heights = list(source2.data['heights'])
            troughs = list(source4.data['troughs'])
            trough_heights = list(source4.data['trough_heights'])
            selected_index = source1.selected.indices[0]
            selected_point = source1.data['t'][selected_index]
            selected_height = source1.data['signal'][selected_index]

            if selected_point in peaks:
                # Remove the peak if selected point is a peak
                index = peaks.index(selected_point)
                peaks.pop(index)
                peak_heights.pop(index)
            elif selected_point in troughs:
                # Remove the trough if selected point is a trough
                index = troughs.index(selected_point)
                troughs.pop(index)
                trough_heights.pop(index)
            else:
                # Add the new peak and its height
                peaks.append(selected_point)
                peak_heights.append(selected_height)

            # Update sources
            source2.data = {'peaks': peaks, 'heights': peak_heights}
            source4.data = {'troughs': troughs, 'trough_heights': trough_heights}

        except IndexError:
            pass  # Handle case where no point is selected

    def change_validity(new):
        global validity
        validity = not validity

    # Create time of death slider
    tf_slider = Slider(title='Time of death', value=initial_tf, start=0, end=(len(signal) - 1), step=1)
    tf_slider.on_change('value', update_tf)

    # Create sliders with on_change callback
    prominence_slider = Slider(title='Prominence', value=initial_prominence, start=0.0, end=1.0, step=0.01)
    prominence_slider.on_change('value', update_peaks)

    height_slider = Slider(title='Height', value=initial_height, start=0.00, end=1.0, step=0.01)
    height_slider.on_change('value', update_peaks)

    bo_slider = Slider(title='Beginning of oscillations', value=initial_tf, start=0, end=(len(signal) - 1), step=1)
    bo_slider.on_change('value', update_bo)
    bo_slider.on_change('value', update_peaks)

    eo_slider = Slider(title='End of oscillations', value=initial_tf, start=0, end=(len(signal) - 1), step=1)
    eo_slider.on_change('value', update_eo)
    eo_slider.on_change('value', update_peaks)

    # Create dropdown menu
    signal_select = Select(title='Select Signal:', value=selected_signal, options=list(signals_df.keys()))
    signal_select.on_change('value', update_signal)

    # Create a button to save data
    save_button = Button(label="Save Data", button_type="success")
    save_button.on_click(save_data)

    # Create a radiobutton to change validity of data
    validity_button = RadioGroup(labels=['Valid', 'Not Valid'], active=0)
    validity_button.on_click(change_validity)

    # Create a checkbox to display BF
    checkbox_display = CheckboxGroup(labels=['Display BF'], active=[0, 1])
    checkbox_display.on_change('active', callback_display)

    # Create a checkbox to display contour
    checkbox_contour = CheckboxGroup(labels=['Plot contour'], active=[0, 1])
    checkbox_contour.on_change('active', callback_contour)

    # Add or remove peaks by tapping
    source1.selected.on_change('indices', tap_point)

    # tap_tool = bokeh.models.TapTool(callback=bokeh.models.CustomJS(args=dict(other_source=source9),code=select_tap_callback()))

    slider_layout = bokeh.layouts.column(
        bokeh.layouts.Spacer(height=30),
        prominence_slider,
        bokeh.layouts.Spacer(height=15),
        height_slider,
        tf_slider,
        bo_slider,
        eo_slider,
    )

    # Dropdown and save button
    dropdown_layout = bokeh.layouts.column(
        bokeh.layouts.Spacer(height=30),
        signal_select,
        save_button,
        checkbox_display,
        checkbox_contour,
        validity_button
    )

    # Set up layout
    norm_layout = bokeh.layouts.row(
        plot,
        bf_display,
        bokeh.layouts.Spacer(width=15),
        slider_layout,
        dropdown_layout,

    )

    # Add layout to the current document
    def norm_app(doc):
        doc.add_root(norm_layout)
    
    bokeh.io.output_notebook()
    bokeh.io.show(norm_app, notebook_url)

    

if __name__ == '__main__':
    path_to_traces = './valid_signals_ppf021_bs_median'    # Path to the traces
    exp_name = 'ppf021'    # Experiment name
    interval = 11    # Sample rate
    bf_folder_path = './bf_images/'
    contours_path = './contour_data_ppf021_median_corr'
    notebook_url = 'localhost:8888'    # Web socket allocated for the Jupyter server (see the web browser url bar)

    # output_folder_path = Path(output_folder)
    # peaks_output_path = Path('./' + output_folder_path.as_posix() + '/peaks_2' + exp_name + '.csv')
    dashboard_registration(traces_path=Path(path_to_traces),
                           exp_name=exp_name,
                           interval=interval,
                           bf_folder_path=bf_folder_path,
                           contours_path=contours_path,
                           notebook_url=notebook_url)

Loading BokehJS ...

In [20]:
peak_data_dashboard

""


First signal


In [33]:
contours_tf = pd.read_csv('./contour_data_ppf021_median_corr', index_col='Time Frame')

In [39]:
contours_tf.loc[(contour_data['Cell Name'] == selected_signal), 'y_coords'][0]

NameError: name 'contour_data' is not defined

In [36]:
contours_std =  pd.read_csv('./contour_data_ppf021_median_corr', index_col=0)

In [37]:
contours_std

,Cell Name,Time Frame,x_coords,y_coords
0,ppf021_xy001_0,0,"[274.0, 275.0, 276.0, 277.0, 278.0, 279.0, 280...","[220.5, 220.5, 220.5, 220.5, 220.5, 220.5, 220..."
1,ppf021_xy001_0,1,"[273.0, 273.5, 274.0, 275.0, 276.0, 277.0, 277...","[222.5, 223.0, 223.5, 223.5, 223.5, 223.5, 223..."
2,ppf021_xy001_0,2,"[267.0, 268.0, 269.0, 270.0, 271.0, 272.0, 272...","[221.5, 221.5, 221.5, 221.5, 221.5, 221.5, 222..."
3,ppf021_xy001_0,3,"[270.0, 271.0, 272.0, 273.0, 274.0, 274.5, 275...","[226.5, 226.5, 226.5, 226.5, 226.5, 227.0, 227..."
4,ppf021_xy001_0,4,"[263.0, 263.5, 264.0, 265.0, 265.5, 266.0, 266...","[219.5, 220.0, 220.5, 220.5, 220.0, 219.5, 220..."
...,...,...,...,...
9911,ppf021_xy110_0,72,"[177.0, 178.0, 179.0, 180.0, 180.5, 181.0, 182...","[78.5, 78.5, 78.5, 78.5, 79.0, 79.5, 79.5, 79...."
9912,ppf021_xy110_0,73,"[175.0, 176.0, 177.0, 178.0, 179.0, 180.0, 181...","[79.5, 79.5, 79.5, 79.5, 79.5, 79.5, 79.5, 79...."
9913,ppf021_xy110_0,74,"[176.0, 177.0, 178.0, 179.0, 180.0, 181.0, 181...","[79.5, 79.5, 79.5, 79.5, 79.5, 79.5, 80.0, 80...."
9914,ppf021_xy110_0,75,"[178.0, 179.0, 179.5, 180.0, 181.0, 182.0, 183...","[78.5, 78.5, 79.0, 79.5, 79.5, 79.5, 79.5, 79...."


In [52]:
contours_tf=contours_tf.iloc[:,1:]

In [58]:
contours_tf.loc[contours_tf['Cell Name']=='ppf021_xy027_0',:].iloc[30:35,:]

,Cell Name,x_coords,y_coords
Time Frame,,,
30,ppf021_xy027_0,"[216.0, 217.0, 217.5, 218.0, 218.5, 219.0, 219...","[159.5, 159.5, 160.0, 160.5, 161.0, 161.5, 162..."
31,ppf021_xy027_0,"[216.0, 217.0, 218.0, 218.5, 219.0, 219.5, 220...","[159.5, 159.5, 159.5, 160.0, 160.5, 161.0, 161..."
32,ppf021_xy027_0,"[231.0, 232.0, 233.0, 233.5, 234.0, 234.5, 235...","[137.5, 137.5, 137.5, 138.0, 138.5, 139.0, 139..."
33,ppf021_xy027_0,"[231.0, 232.0, 232.5, 233.0, 233.5, 234.0, 234...","[137.5, 137.5, 138.0, 138.5, 139.0, 139.5, 140..."
34,ppf021_xy027_0,"[215.0, 216.0, 217.0, 217.5, 218.0, 218.5, 219...","[159.5, 159.5, 159.5, 160.0, 160.5, 161.0, 161..."


In [63]:
print(contours_tf.loc[contours_tf['Cell Name']=='ppf021_xy027_0',:].iloc[32,2])

[137.5, 137.5, 137.5, 138.0, 138.5, 139.0, 139.5, 140.0, 141.0, 142.0, 142.5, 143.0, 144.0, 144.5, 145.0, 146.0, 146.5, 147.0, 148.0, 149.0, 149.5, 150.0, 150.5, 151.0, 151.5, 152.0, 153.0, 153.5, 154.0, 154.5, 155.0, 156.0, 156.5, 157.0, 157.5, 158.0, 158.5, 159.0, 160.0, 160.5, 161.0, 161.5, 162.0, 163.0, 163.5, 164.0, 165.0, 166.0, 166.5, 167.0, 168.0, 168.5, 169.0, 169.5, 169.5, 170.0, 170.5, 170.0, 169.5, 170.0, 170.5, 171.0, 171.5, 172.0, 172.5, 173.0, 174.0, 174.5, 175.0, 176.0, 176.5, 177.0, 177.5, 178.0, 178.5, 179.0, 179.5, 180.0, 180.5, 181.0, 181.5, 182.0, 182.5, 183.0, 183.5, 184.0, 184.5, 185.0, 185.5, 186.0, 187.0, 187.5, 188.0, 189.0, 189.5, 190.0, 191.0, 191.5, 192.0, 192.5, 193.0, 193.5, 194.0, 194.5, 195.0, 195.5, 195.5, 196.0, 196.5, 196.5, 196.5, 196.5, 196.5, 196.5, 197.0, 197.5, 197.5, 198.0, 198.5, 198.5, 199.0, 199.5, 200.0, 200.5, 200.0, 199.5, 200.0, 200.5, 200.0, 199.5, 199.0, 198.5, 198.0, 197.5, 197.0, 196.0, 195.0, 194.0, 193.5, 193.0, 192.0, 191.0, 190.5

In [65]:
print(contours_tf.loc[contours_tf['Cell Name']=='ppf021_xy027_0',:].iloc[31,2])

[159.5, 159.5, 159.5, 160.0, 160.5, 161.0, 161.5, 162.0, 163.0, 163.5, 164.0, 164.5, 165.0, 165.5, 165.5, 166.0, 166.5, 166.5, 166.0, 165.5, 166.0, 166.5, 166.5, 166.0, 165.5, 166.0, 166.5, 167.0, 167.5, 168.0, 168.5, 169.0, 170.0, 171.0, 171.5, 172.0, 173.0, 173.5, 174.0, 174.5, 175.0, 175.5, 176.0, 176.5, 177.0, 178.0, 178.5, 179.0, 180.0, 180.5, 181.0, 182.0, 182.5, 183.0, 183.5, 184.0, 185.0, 186.0, 186.5, 187.0, 188.0, 188.5, 189.0, 190.0, 190.5, 191.0, 191.5, 192.0, 193.0, 193.5, 194.0, 195.0, 195.5, 196.0, 197.0, 197.5, 198.0, 198.5, 198.5, 199.0, 199.5, 199.5, 199.5, 199.5, 200.0, 200.5, 201.0, 201.5, 202.0, 203.0, 204.0, 204.5, 205.0, 205.5, 206.0, 206.5, 206.5, 206.5, 206.0, 205.5, 205.0, 204.5, 204.0, 203.5, 203.0, 202.5, 202.5, 202.0, 201.5, 202.0, 202.5, 202.5, 203.0, 203.5, 204.0, 204.5, 205.0, 205.5, 205.5, 206.0, 206.5, 207.0, 207.5, 208.0, 208.5, 208.5, 208.5, 208.0, 207.5, 207.5, 207.5, 208.0, 208.5, 208.5, 208.5, 208.0, 207.5, 207.5, 207.0, 206.5, 206.5, 206.0, 205.5

In [71]:
len(contours_tf.loc[contours_tf['Cell Name']=='ppf021_xy027_0',:].iloc[32,1])

2051

In [77]:
contours_tf

,Cell Name,x_coords,y_coords
Time Frame,,,
0,ppf021_xy001_0,"[274.0, 275.0, 276.0, 277.0, 278.0, 279.0, 280...","[220.5, 220.5, 220.5, 220.5, 220.5, 220.5, 220..."
1,ppf021_xy001_0,"[273.0, 273.5, 274.0, 275.0, 276.0, 277.0, 277...","[222.5, 223.0, 223.5, 223.5, 223.5, 223.5, 223..."
2,ppf021_xy001_0,"[267.0, 268.0, 269.0, 270.0, 271.0, 272.0, 272...","[221.5, 221.5, 221.5, 221.5, 221.5, 221.5, 222..."
3,ppf021_xy001_0,"[270.0, 271.0, 272.0, 273.0, 274.0, 274.5, 275...","[226.5, 226.5, 226.5, 226.5, 226.5, 227.0, 227..."
4,ppf021_xy001_0,"[263.0, 263.5, 264.0, 265.0, 265.5, 266.0, 266...","[219.5, 220.0, 220.5, 220.5, 220.0, 219.5, 220..."
...,...,...,...
72,ppf021_xy110_0,"[177.0, 178.0, 179.0, 180.0, 180.5, 181.0, 182...","[78.5, 78.5, 78.5, 78.5, 79.0, 79.5, 79.5, 79...."
73,ppf021_xy110_0,"[175.0, 176.0, 177.0, 178.0, 179.0, 180.0, 181...","[79.5, 79.5, 79.5, 79.5, 79.5, 79.5, 79.5, 79...."
74,ppf021_xy110_0,"[176.0, 177.0, 178.0, 179.0, 180.0, 181.0, 181...","[79.5, 79.5, 79.5, 79.5, 79.5, 79.5, 80.0, 80...."


In [133]:
for signal in contours_tf['Cell Name'].unique():
    mask = contours_tf['Cell Name'] == signal
    # print(signal)
    for tf in contours_tf.loc[mask,:].index:
        if len(contours_tf.loc[mask,:].iloc[tf,1])<400:
            print(signal, tf)
        
    

ppf021_xy019_0 25
ppf021_xy019_0 28
ppf021_xy019_0 29
ppf021_xy019_0 30
ppf021_xy019_0 36
ppf021_xy019_0 40
ppf021_xy019_0 46
ppf021_xy019_0 47
ppf021_xy019_0 51
ppf021_xy019_0 57
ppf021_xy019_0 60
ppf021_xy019_0 62
ppf021_xy019_0 70
ppf021_xy019_0 71
ppf021_xy019_0 73
ppf021_xy019_0 74


IndexError: index 65 is out of bounds for axis 0 with size 65

In [79]:
contours_tf.index

Int64Index([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9,
            ...
            67, 68, 69, 70, 71, 72, 73, 74, 75, 76],
           dtype='int64', name='Time Frame', length=9916)

In [100]:
mask = contours_tf['Cell Name'] == 'ppf021_xy027_0'

for tf in contours_tf.loc[mask,:].index:
        if len(contours_tf.loc[mask,:].iloc[tf,1])<400:
            print(tf)
        

33


In [134]:
for signal in contours_tf['Cell Name'].unique():
    mask = contours_tf['Cell Name'] == signal
    for tf in contours_tf.loc[mask,:].index:
        if len(ast.literal_eval(contours_tf.loc[(mask), 'x_coords'][tf]))<55:
            print(signal, tf)
        

ppf021_xy019_0 25
ppf021_xy019_0 28
ppf021_xy019_0 29
ppf021_xy019_0 30
ppf021_xy019_0 36
ppf021_xy019_0 40
ppf021_xy019_0 46
ppf021_xy019_0 47
ppf021_xy019_0 51
ppf021_xy019_0 57
ppf021_xy019_0 60
ppf021_xy019_0 62
ppf021_xy019_0 70
ppf021_xy019_0 71
ppf021_xy019_0 73
ppf021_xy019_0 74
ppf021_xy025_0 17
ppf021_xy025_0 23
ppf021_xy025_0 25
ppf021_xy025_0 47
ppf021_xy025_0 61
ppf021_xy027_0 33
ppf021_xy089_1 20


In [89]:
contours_tf.loc[contours_tf['Cell Name'] == 'ppf021_xy027_0',:].iloc[[32,33],1]

Time Frame
32    [231.0, 232.0, 233.0, 233.5, 234.0, 234.5, 235...
33    [231.0, 232.0, 232.5, 233.0, 233.5, 234.0, 234...
Name: x_coords, dtype: object

In [91]:
print(contours_tf.loc[contours_tf['Cell Name'] == 'ppf021_xy027_0',:].iloc[32,1])

[231.0, 232.0, 233.0, 233.5, 234.0, 234.5, 235.0, 235.5, 235.5, 235.5, 235.0, 234.5, 234.5, 235.0, 235.5, 235.5, 236.0, 236.5, 236.5, 236.5, 236.0, 235.5, 235.0, 234.5, 234.0, 233.5, 233.5, 233.0, 232.5, 233.0, 233.5, 233.5, 234.0, 234.5, 235.0, 235.5, 236.0, 236.5, 236.5, 237.0, 237.5, 238.0, 238.5, 238.5, 238.0, 237.5, 237.5, 237.5, 238.0, 238.5, 238.5, 239.0, 239.5, 240.0, 241.0, 241.5, 242.0, 242.5, 243.0, 243.5, 244.0, 244.5, 245.0, 245.5, 245.0, 244.5, 244.5, 244.0, 243.5, 243.5, 243.0, 242.5, 242.0, 241.5, 241.0, 240.5, 240.0, 239.5, 239.0, 238.5, 238.0, 237.5, 237.0, 236.5, 236.0, 235.5, 235.0, 234.5, 234.0, 233.5, 233.5, 233.0, 232.5, 232.5, 232.0, 231.5, 231.5, 231.0, 230.5, 230.0, 229.5, 229.0, 228.5, 228.0, 227.5, 227.0, 226.0, 225.5, 225.0, 224.0, 223.0, 222.0, 221.0, 220.0, 219.5, 219.0, 218.0, 217.5, 217.0, 216.0, 215.5, 215.0, 214.5, 214.0, 213.5, 213.0, 212.5, 212.0, 211.5, 211.0, 210.5, 210.0, 209.5, 209.0, 208.5, 208.5, 208.5, 208.5, 208.0, 207.5, 207.5, 207.5, 207.0

In [92]:
print(contours_tf.loc[contours_tf['Cell Name'] == 'ppf021_xy027_0',:].iloc[33,1])

[231.0, 232.0, 232.5, 233.0, 233.5, 234.0, 234.5, 235.0, 235.5, 235.5, 235.0, 234.5, 234.5, 234.5, 235.0, 235.5, 235.5, 236.0, 236.5, 236.5, 236.0, 235.5, 235.0, 234.5, 234.0, 233.5, 233.5, 233.0, 232.5, 232.0, 231.5, 231.5, 231.0, 230.5, 230.0, 229.5, 229.0, 228.5, 229.0, 229.5, 229.5, 229.5, 229.5, 229.0, 228.5, 228.5, 228.5, 228.5, 229.0, 229.5, 230.0, 230.5, 231.0]


In [102]:
len(contours_tf.loc[contours_tf['Cell Name'] == 'ppf021_xy027_0',:].iloc[33,1])

371

In [112]:
n_points = len(contours_tf.loc[contours_tf['Cell Name'] == 'ppf021_xy027_0',:].iloc[33,1])/7

In [113]:
n_points

53.0

In [114]:
n_points = len(contours_tf.loc[contours_tf['Cell Name'] == 'ppf021_xy027_0',:].iloc[32,1])/7

In [115]:
n_points

293.0

In [122]:
len(ast.literal_eval(contours_tf.loc[(contours_tf['Cell Name'] == 'ppf021_xy027_0'), 'x_coords'][33]))

53

In [126]:
len(contours_tf.loc[(contours_tf['Cell Name'] == 'ppf021_xy027_0'),'x_coords'])

77

In [ ]:
for signal in contours_tf['Cell Name'].unique():
    mask = contours_tf['Cell Name'] == signal
    print(signal)
    for tf in contours_tf.loc[mask,:].index:
        if len(contours_tf.loc[mask,:].iloc[tf,1])<400:
            print(signal, tf)
        
    